# <center><font color="blue"> DSM pipeline using GEE in Python API </font></center>
Adapted by Professor Raul Poppiel from the geospatial mapping pipeline of van den Hoogen et al.: [Code](https://github.com/hooge104/geospatial_mapping_pipeline) (MIT License, Copyright (c) 2021 hooge104) & [Paper](https://doi.org/10.1101/2021.07.07.451145)

EE Code [examples](https://github.com/google/earthengine-api/tree/master/python/examples/ipynb) and [Tutorials](https://github.com/giswqs/earthengine-py-notebooks) using Jupyter Notebook (.ipynb)

Markdown [syntax](https://www.markdownguide.org/basic-syntax/) and [hacks](https://www.markdownguide.org/hacks/)

Studies using the original pipeline: [1](https://doi.org/10.1111/gcb.16060),  [2](https://doi.org/10.5194/essd-14-4339-2022),  [3](https://doi.org/10.1111/gcb.15892)

**Before running:** edit every value marked `USER SETTING` (Google Cloud project in section 1; account, file and folder settings in sections 2 and 3).

## 1) Import the modules of interest

In [ ]:
# Import earthengine api (and authenticate, if necessary)
import ee
ee.Authenticate(force=True)  # re-authentication may be required (every 7 days)

In [ ]:
# Install the dependencies (uncomment if needed; see requirements.txt in the repository)
# !pip install -r requirements.txt
# or, individually:
# !pip install earthengine-api google-cloud-storage "pandas>=2.0,<3.0" numpy scipy scikit-learn folium geehydro


In [ ]:
# Import the modules of interest

import pandas as pd
import numpy as np
import subprocess
import time
import datetime
import os
import re
from functools import partial
from pathlib import Path
from scipy.spatial import ConvexHull
from sklearn.decomposition import PCA
from itertools import combinations
from itertools import repeat
from pathlib import Path
import folium
import geehydro
from google.cloud import storage
#from google.colab import auth
# Google Cloud project registered for Earth Engine
projectIdGEE = 'your-gee-cloud-project'  # USER SETTING
ee.Initialize(project=projectIdGEE)

## 2) Project-specific settings

In [ ]:
# Root of your GEE assets  # USER SETTING
# !! Legacy roots ('users/<username>') can no longer be created by new Earth Engine accounts.
# !! New users: use a Cloud project root, 'projects/<your-gee-cloud-project>/assets' (default below).
# !! Existing legacy users may keep 'users/<username>' (the 'users/' prefix is required);
# !! the asset structure below the root is the same.
assetRootGEE = 'projects/'+projectIdGEE+'/assets'
if not assetRootGEE.startswith(('users/', 'projects/')):
    raise ValueError("assetRootGEE must start with 'users/' (legacy root) or 'projects/' (Cloud project root): "+assetRootGEE)

# Your Cloud Storage Bucket name  # USER SETTING
# !! This bucket should be pre-created before running this script
# !! It is used only as temporary storage for the bootstrap table (see "Bootstrapping"):
# !! the file is deleted from the bucket as soon as Earth Engine has ingested it.
# !! Disable soft delete on this bucket; otherwise deleted files are retained (and billed) for 7 days.
GCBbucketname = 'your-gcs-bucket'
# Optional folder inside the bucket where the temporary file is written  # USER SETTING
# (e.g., 'folder/subfolder'); leave '' to write at the bucket root.
# !! The bucket name above must not contain '/': in 'gs://my-bucket/folder/subfolder',
# !! the bucket is 'my-bucket' and the folder is 'folder/subfolder'.
GCSfolder = ''
if '/' in GCBbucketname:
    raise ValueError("GCBbucketname must contain only the bucket name; put the folder path in GCSfolder: "+GCBbucketname)

# Specify file name of raw point collection (without extension); must be a csv file; don't include '.csv'
titleOfRawPointCollection = 'DB_C_g_kg_atr_90'  # USER SETTING (e.g., 'RC_LUCAS_C_gkg' for the example dataset)

# Input the name of the classification property
classProperty = 'C_gkg'  # USER SETTING
# Input the number of decimal places to round the classification property values (of raw point collection)
classProperty_roundTo = 2 #decimal places

# Input the name of the project folder inside GEE which all of the assets will be stored
# This folder will be generated automatically in GEE
projectFolder = 'RC_C_PROT3'  # USER SETTING (one folder per mapped attribute)

# Specify the column names where the latitude and longitude information is stored: these columns must be present in the csv containing the observations
latString = 'Y'
longString = 'X'

# Path to COLAB directory  holding input data  # USER SETTING
holdingFolder = './data'

# Path to COLAB direcotry for output data  # USER SETTING
outputFolder = './data/Maps'

# Create directory to hold training data
Path(outputFolder).mkdir(parents=True, exist_ok=True)

# Local path of earthengine-api (ee) and gsutil python dependencies (packages)
# USER SETTING: replace with the full path to the executables if they are not on the system PATH
bashFunction_EarthEngine = 'earthengine'
bashFunctionGSUtil = 'gsutil'

# Perform modeling in log space? (True or False)
log_transform_classProperty = False

# Ensemble of top 10 models from grid search? (True or False) --> if False: Get top model (only one)
ensemble = True

# Proportion of variance to be covered by the PCA for interpolation/extrapolation
propOfVariance = 90

In [ ]:
# Play a beep sound when a task is completed
from IPython.display import Audio
import numpy as np

beep = np.sin(2*np.pi*1000*np.arange(5000*2)/10000)
notify=Audio(beep, rate=10000, autoplay=True)
notify

## 3) DSM pipeline settings

### Export settings

In [ ]:
# Set pyramidingPolicy for exporting purposes
pyramidingPolicy = 'mean'

# Specify CRS to use (of both raw csv and final maps)
CRStoUse = 'EPSG:4326'

# Set resolution of final image in arc seconds (1 arc seconds equals to ± 30m at the equator)
export_res = 90  # in meters

# Convert resolution to degrees (https://opendem.info/arc2meters.html)
res_deg = export_res/111319.49079327  # 1 degree = 111319.49079327 m at Equator

# Dfine the size (in pixels) of the tiles in which the final image will be computed (exported)
shardSize = 128 #64 #128  #256

# Geometry to use for export

exportingGeometry = ee.Geometry.Polygon([[[10.59775055802253,47.45994676713431], [20.155856026772533,47.45994676713431], [20.155856026772533,52.1758711141354],
                                          [10.59775055802253,52.1758711141354], [10.59775055802253,47.45994676713431]]], None, False);

In [ ]:
# Create an interactive map

# Get the center coordinates of the BBox
centroid = exportingGeometry.bounds().centroid(1).getInfo()["coordinates"];
lat = centroid[1]; lon = centroid[0];

# Center the map and display the BBox
Map = folium.Map(tiles='OpenStreetMap')
Map.setOptions()

Map.addLayer(exportingGeometry, {}, 'BBox')
Map.setCenter(lon,lat,2)

Map.setControlVisibility()
Map

### General settings

In [ ]:
# Input the normal wait time (in seconds) for "wait and break" cells
normalWaitTime = 5

### Covariate data settings
GEE covariate [examples](https://gee-community-catalog.org/)

In [ ]:
# Covariate set used in this study

# List of the covariates to use (enable/disable using #)
covariateList = [
"blue", "green", "red", "nir", "swir1", "swir2", #--> SoilVegetation Information (SYSI)
"Elevation", "Slope", "Aspect", "Hillshade", "Northness", "Eastness", "HorizontalCurvature", "VerticalCurvature", "MeanCurvature", "GaussianCurvature", "MinimalCurvature", "MaximalCurvature", #"ShapeIndex", #--> TAGEE: https://doi.org/10.3390/ijgi9060400
"MultiScaleTopographicPositionIndex", #--> https://doi.org/10.1371/journal.pone.0143619
'EuclideanDistance_toWater',
"CHELSA_BIO_Precipitation_Seasonality", "CHELSA_BIO_Temperature_Annual_Range",
"CHELSA_BIO_Annual_Mean_Temperature", "CHELSA_BIO_Annual_Precipitation",
]

# Load the composite on which to perform the mapping, and subselect the bands of interest
Climate = ee.Image("projects/ee-geocisworldspecs-203/assets/CHELSA_BIO_RC_90m").clip(exportingGeometry)


SoilVegetation = ee.Image('projects/ee-geocisworldspecs-203/assets/SyVI_RC_1984_2023')


Relief = ee.Image.cat([ee.Image('projects/ee-geocisworldspecs-203/assets/TAGEE_RC_30m'),
                       ee.Image('projects/ee-geocisworldspecs-203/assets/MultiScaleTopographicPositionIndex_RC_90m')]).clip(exportingGeometry)


Location = ee.Image.cat([ee.Image('projects/ee-geocisworldspecs-203/assets/EuclideanDistance_toWater_RC_90m_ORD_STRA_3')]).clip(exportingGeometry)


Resolve_Biome = ee.Image("projects/ee-geocisworldspecs-203/assets/example_composite_30ArcSec_RC_90m").select(['Resolve_Biome']).clip(exportingGeometry)


full_composite = ee.Image.cat(SoilVegetation,Climate,Relief,Location,Resolve_Biome) # scorpan

compositeToClassify = full_composite.select(covariateList)
numberOfCovariates = compositeToClassify.bandNames().size().getInfo()
print('- Covariate Nº:',numberOfCovariates)
print('- Covariate names:',compositeToClassify.bandNames().getInfo())

# Scale of composite
scale = SoilVegetation.projection().nominalScale().getInfo()
print('- Pixel size of composite is '+str(scale)+'m')

In [ ]:
# Center the map
Map = folium.Map(tiles='OpenStreetMap')
Map.setOptions()

# Visualization parameters
rgbTrueColor = {'bands': ['red', 'green', 'blue'], 'min': [550,570,500], 'max': [1900,1500,1100], 'gamma': 1};
rgbFalseColor = {'bands': ['swir1', 'nir', 'red'], 'min': [950,1250,300], 'max': [3600,3500,1600], 'gamma': 1};

# Display the Image
Map.addLayer(full_composite, rgbTrueColor, '90m Soil-Vegetation image 4 RGB 321', False)
Map.addLayer(full_composite, rgbFalseColor, '90m Soil-Vegetation image 4 RGB 543')

Map.setCenter(lon,lat,2)

Map.setControlVisibility()
Map

### Additional settings
### RF and Cross validation settings

In [ ]:
# Grid search parameters; specify range
# variables per split
varsPerSplit_list = list(range(4, numberOfCovariates+1, 4))

# minium leaf population
leafPop_list = list([3,6,9])

# maximum number of leaf nodes in each tree (controls the tree size)
maxNodes_list = list([500,1000,1500])

num_models = len(varsPerSplit_list)*len(leafPop_list)*len(maxNodes_list)
print('Grid search combinations: ', num_models, 'models')
print(' -varsPerSplit_list: ',varsPerSplit_list), print(' -leafPop_list:      ',leafPop_list), print(' -maxNodes_list:     ',maxNodes_list)

# Set k for k-fold CV and make a list of the k-fold CV assignments to use
k = 10
kList = list(range(1,k+1))

# Metric to use for sorting k-fold CV hyperparameter tuning
sort_acc_prop = 'Mean_R2' # (either one of 'Mean_R2', 'Mean_MAE', 'Mean_RMSE')

# Set number of trees in RF models
nTrees = 20

# Spatial leave-one-out cross-validation settings
# skip test points outside training space after removing points in buffer zone? This might reduce extrapolation but overestimate accuracy
loo_cv_wPointRemoval = False

# Input the name of the property that holds the CV fold assignment
cvFoldString = 'CV_Fold'

# Input the title of the CSV that will hold all of the data that has been given a CV fold assignment
titleOfCSVWithCVAssignments = classProperty+'_training_data' #_SpatialFiltered' #_export'

assetIDForCVAssignedColl = assetRootGEE+'/'+projectFolder+'/'+classProperty+'_training_data' #_SpatialFiltered_export'

### Bootstrap settings

In [ ]:
# Number of bootstrap iterations
bootstrapIterations = 100

# Generate the seeds for bootstrapping
seedsToUseForBootstrapping = list(range(1, bootstrapIterations+1))

# Input the header text that will name the bootstrapped dataset
bootstrapSamples = classProperty+'_bootstrapSamples'

# Input the dictionary of values for each of the stratification category levels
# !! This area breakdown determines the proportion of each biome (%) to include in every bootstrap
# Recalculate the proportions according to your study area

strataDict = {
# these values must be ADJUSTED (calculated) to your area using the following code
# ---> https://code.earthengine.google.com/88f7475aeb53873bec72b7ff31379c75
1: 0.00,
2: 0.00,
3: 0.00,
4: 90.91,
5: 9.09,
6: 0.00,
7: 0.00,
8: 0.00,
9: 0.00,
10: 0.00,
11: 0.00,
12: 0.00,
13: 0.00,
14: 0.00
} #Total = 100%

# Stratification inputs
# Write the name of the variable used for stratification
# !! This variable should be included in the input dataset
stratificationVariableString = 'Resolve_Biome'   # Reference: https://doi.org/10.1093/biosci/bix014

# Center the map and display the Stratification Variable
Map = folium.Map(tiles='OpenStreetMap')
Map.setOptions()

Map.addLayer(full_composite.select(stratificationVariableString)
             .clip(exportingGeometry), {'min':1,'max':14}, stratificationVariableString)
Map.setCenter(lon,lat,2)

Map.setControlVisibility()
Map

### Bash and Google Cloud Bucket settings

In [ ]:
# Specify the necessary arguments to upload the files to a Cloud Storage bucket
# I.e., create bash variables in order to create/check/delete Earth Engine Assets

# Specify the arguments to these functions
arglist_preEEUploadTable = ['upload','table']
arglist_initEEUploadTable = ['--x_column', longString, '--y_column', latString, '--crs', CRStoUse]
arglist_postEEUploadTable = ['--x_column', 'Pixel_Long', '--y_column', 'Pixel_Lat']
arglist_preGSUtilUploadFile = ['cp']
formattedBucketOI = 'gs://'+GCBbucketname+('/'+GCSfolder.strip('/') if GCSfolder.strip('/') else '')
assetIDStringPrefix = '--asset_id='
arglist_CreateCollection = ['create','collection']
arglist_CreateFolder = ['create','folder']
arglist_Detect = ['asset','info']
arglist_ls = ['ls']
arglist_Delete = ['rm','-r']
stringsOfInterest = ['Asset does not exist or is not accessible']

# Prefix for earthengine command-line calls; the Cloud project is passed explicitly
bashPrefix_EarthEngine = [bashFunction_EarthEngine, '--project', projectIdGEE]

# Compose the arguments into lists that can be run via the subprocess module
bashCommandList_Detect = bashPrefix_EarthEngine+arglist_Detect
bashCommandList_ls = bashPrefix_EarthEngine+arglist_ls
bashCommandList_Delete = bashPrefix_EarthEngine+arglist_Delete
bashCommandList_CreateCollection = bashPrefix_EarthEngine+arglist_CreateCollection
bashCommandList_CreateFolder = bashPrefix_EarthEngine+arglist_CreateFolder

# Helper functions for uploading tables to Earth Engine and monitoring tasks
def asset_exists(assetId):
    """Return True if the Earth Engine asset exists and is accessible."""
    try:
        ee.data.getAsset(assetId)
        return True
    except ee.EEException:
        return False

def wait_for_task(taskId):
    """Wait for an Earth Engine task (export or ingestion) and stop with a clear message if it fails."""
    while True:
        status = ee.data.getTaskStatus(taskId)[0]
        state = status.get('state')
        if state in ('COMPLETED', 'SUCCEEDED'):
            print('\n', status.get('description', taskId), 'completed.')
            return
        if state in ('FAILED', 'CANCELLED', 'UNKNOWN'):
            raise RuntimeError('Task '+str(status.get('description', taskId))+' ended with state '+str(state)+': '+str(status.get('error_message', '')))
        print(datetime.datetime.fromtimestamp(time.time()).strftime('%Y-%m-%d %H:%M:%S'), 'Task state:', state, end='\r')
        time.sleep(normalWaitTime)

def df_to_fc(df):
    """Convert a DataFrame with longitude/latitude columns into an ee.FeatureCollection of points.
    All columns are kept as feature properties, as in a manual CSV upload."""
    features = []
    for record in df.to_dict('records'):
        props = {k: (v.item() if hasattr(v, 'item') else v) for k, v in record.items()}
        features.append(ee.Feature(ee.Geometry.Point([props[longString], props[latString]]), props))
    return ee.FeatureCollection(features)

def export_fc_to_asset(fc, description, assetId):
    """Start an export of a FeatureCollection to an Earth Engine table asset and return the task ID."""
    task = ee.batch.Export.table.toAsset(collection=fc, description=description, assetId=assetId)
    task.start()
    return task.id


### Helper functions

###### Performance metrics

In [ ]:
# Function to convert FeatureCollection to Image
def fcToImg(f):
	# Reduce to image, take mean per pixel
	img = sampledFC.reduceToImage(
		properties = [f],
		reducer = ee.Reducer.mean()
	)
	return img

# Function to convert GEE FC to pd.DataFrame. Not ideal as it's calling .getInfo(), but does the job
def GEE_FC_to_pd(fc):
	result = []
	# Fetch data as a list
	values = fc.toList(100000).getInfo()
	# Fetch column names
	BANDS = fc.first().propertyNames().getInfo()
	# Remove system:index if present
#	if 'system:index' in BANDS: BANDS.remove('system:index')
	BANDS.remove('system:index')

	# Convert to data frame
	for item in values:
		values = item['properties']
		row = [str(values[key]) for key in BANDS]
		row = ",".join(row)
		result.append(row)

	df = pd.DataFrame([item.split(",") for item in result], columns = BANDS)
	df.replace('None', np.nan, inplace = True)

	return df

# R^2 function
def coefficientOfDetermination(fcOI,propertyOfInterest,propertyOfInterest_Predicted):
	# Compute the mean of the property of interest
	propertyOfInterestMean = ee.Number(ee.Dictionary(ee.FeatureCollection(fcOI).select([propertyOfInterest]).reduceColumns(ee.Reducer.mean(),[propertyOfInterest])).get('mean'));

	# Compute the total sum of squares
	def totalSoSFunction(f):
		return f.set('Difference_Squared',ee.Number(ee.Feature(f).get(propertyOfInterest)).subtract(propertyOfInterestMean).pow(ee.Number(2)))
	totalSumOfSquares = ee.Number(ee.Dictionary(ee.FeatureCollection(fcOI).map(totalSoSFunction).select(['Difference_Squared']).reduceColumns(ee.Reducer.sum(),['Difference_Squared'])).get('sum'))

	# Compute the residual sum of squares
	def residualSoSFunction(f):
		return f.set('Residual_Squared',ee.Number(ee.Feature(f).get(propertyOfInterest)).subtract(ee.Number(ee.Feature(f).get(propertyOfInterest_Predicted))).pow(ee.Number(2)))
	residualSumOfSquares = ee.Number(ee.Dictionary(ee.FeatureCollection(fcOI).map(residualSoSFunction).select(['Residual_Squared']).reduceColumns(ee.Reducer.sum(),['Residual_Squared'])).get('sum'))

	# Finalize the calculation
	r2 = ee.Number(1).subtract(residualSumOfSquares.divide(totalSumOfSquares))

	return ee.Number(r2)

# RMSE function
def RMSE(fcOI,propertyOfInterest,propertyOfInterest_Predicted):
	# Compute the squared difference between observed and predicted
	def propDiff(f):
		diff = ee.Number(f.get(propertyOfInterest)).subtract(ee.Number(f.get(propertyOfInterest_Predicted)))

		return f.set('diff', diff.pow(2))

	# calculate RMSE from squared difference
	rmse = ee.Number(fcOI.map(propDiff).reduceColumns(ee.Reducer.mean(), ['diff']).get('mean')).sqrt()

	return rmse

## MAE function
#def MAE(fcOI,propertyOfInterest,propertyOfInterest_Predicted):
#	# Compute the absolute difference between observed and predicted
#	def propDiff(f):
#		diff = ee.Number(f.get(propertyOfInterest)).subtract(ee.Number(f.get(propertyOfInterest_Predicted)))
#
#		return f.set('diff', diff.abs())
#
#	# calculate MAE from difference
#	mae = ee.Number(fcOI.map(propDiff).reduceColumns(ee.Reducer.mean(), ['diff']).get('mean'))
#
#	return mae

# RPIQ function
def RPIQ(fcOI,propertyOfInterest,propertyOfInterest_Predicted):
	# Compute the squared difference between observed and predicted
	def propDiff(f):
		diff = ee.Number(f.get(propertyOfInterest)).subtract(ee.Number(f.get(propertyOfInterest_Predicted)))

		return f.set('diff', diff.pow(2))

	# calculate RMSE from squared difference
	rmse = ee.Number(fcOI.map(propDiff).reduceColumns(ee.Reducer.mean(), ['diff']).get('mean')).sqrt()

	# Compute the interquartile range
	Q1 = ee.Number(ee.Dictionary(ee.FeatureCollection(fcOI).select([propertyOfInterest]).reduceColumns(ee.Reducer.percentile([25], ['q1']),[propertyOfInterest])).get('q1'))
	Q3 = ee.Number(ee.Dictionary(ee.FeatureCollection(fcOI).select([propertyOfInterest]).reduceColumns(ee.Reducer.percentile([75], ['q3']),[propertyOfInterest])).get('q3'))
	iqr = Q3.subtract(Q1)

	# calculate RPIQ from squared difference and interquartile range
	rpiq = iqr.divide(rmse)

	return ee.Number(rpiq)

# Function to add folds stratified per biome
def assignFolds(biome):
	fc_filtered = fc_agg.filter(ee.Filter.eq(stratificationVariableString, biome))

	cvFoldsToAssign = ee.List.sequence(0, fc_filtered.size()).map(lambda i: ee.Number(i).mod(k).add(1))

	fc_sorted = fc_filtered.randomColumn(seed = biome).sort('random')

	fc_wCVfolds = ee.FeatureCollection(cvFoldsToAssign.zip(fc_sorted.toList(fc_filtered.size())).map(lambda f: ee.Feature(ee.List(f).get(1)).set(cvFoldString, ee.List(f).get(0))))

	return fc_wCVfolds

###### Cross-Validation (compute mean values)

In [ ]:
# Define a function to take a feature with a classifier of interest
def computeCVAccuracyAndRMSE_mean(featureWithClassifier):
	# Pull the classifier from the feature
	cOI = ee.Classifier(featureWithClassifier.get('c'))

	# Create a function to map through the fold assignments and compute the overall accuracy
	# for all validation folds
	def computeAccuracyForFold(foldFeature):
		# Organize the training and validation data
		foldNumber = ee.Number(ee.Feature(foldFeature).get('Fold'))
		trainingData = fcOI.filterMetadata(cvFoldString,'not_equals',foldNumber)
		validationData = fcOI.filterMetadata(cvFoldString,'equals',foldNumber)

		# Train the classifier and classify the validation dataset
#		trainedClassifier = cOI.train(trainingData,classProperty,covariateList, 0.2, 1).train(trainingData,classProperty,covariateList, 0.2, 2).train(trainingData,classProperty,covariateList, 0.2, 3).train(trainingData,classProperty,covariateList, 0.2, 4).train(trainingData,classProperty,covariateList, 0.2, 5)
		trainedClassifier = cOI.train(trainingData,classProperty,covariateList)
		outputtedPropName = classProperty+'_Predicted'
		classifiedValidationData = validationData.classify(trainedClassifier,outputtedPropName)

		# Compute accuracy metrics
		r2ToSet = coefficientOfDetermination(classifiedValidationData,classProperty,outputtedPropName)
		rmseToSet = RMSE(classifiedValidationData,classProperty,outputtedPropName)
#		maeToSet = MAE(classifiedValidationData,classProperty,outputtedPropName)
		rpiqToSet = RPIQ(classifiedValidationData,classProperty,outputtedPropName)
		return foldFeature.set('R2',r2ToSet).set('RMSE', rmseToSet).set('RPIQ', rpiqToSet)#.set('MAE', maeToSet)

	# Compute the mean and std dev of the accuracy values of the classifier across all folds
	accuracyFC = kFoldAssignmentFC.map(computeAccuracyForFold)
	meanAccuracy = accuracyFC.aggregate_mean('R2')
#	sdAccuracy = accuracyFC.aggregate_total_sd('R2')

	# Calculate mean and std dev of RMSE
	RMSEvals = accuracyFC.aggregate_array('RMSE')
	RMSEvalsSquared = RMSEvals.map(lambda f: ee.Number(f).multiply(f))
	sumOfRMSEvalsSquared = RMSEvalsSquared.reduce(ee.Reducer.sum())
	meanRMSE = ee.Number.sqrt(ee.Number(sumOfRMSEvalsSquared).divide(k))

#	sdRMSE = accuracyFC.aggregate_total_sd('RMSE')

#	# Calculate mean and std dev of MAE
#	meanMAE = accuracyFC.aggregate_mean('MAE')
#	sdMAE= accuracyFC.aggregate_total_sd('MAE')

	# Calculate mean and std dev of RPIQ
	meanRPIQ = accuracyFC.aggregate_mean('RPIQ')
#	sdRPIQ= accuracyFC.aggregate_total_sd('RPIQ')

	# Compute the feature to return
#	featureToReturn = featureWithClassifier.select(['cName']).set('Mean_R2',meanAccuracy,'StDev_R2',sdAccuracy, 'Mean_RMSE',meanRMSE,'StDev_RMSE',sdRMSE, 'Mean_RPIQ',meanRPIQ,'StDev_RPIQ',sdRPIQ) # , 'Mean_MAE',meanMAE,'StDev_MAE',sdMAE
	featureToReturn = featureWithClassifier.select(['cName']).set('Mean_R2',meanAccuracy,'Mean_RMSE',meanRMSE,'Mean_RPIQ',meanRPIQ) # , 'Mean_MAE',meanMAE,'StDev_MAE',sdMAE
	return featureToReturn

###### Cross-Validation (compute standard deviation values)

In [ ]:
# Define a function to take a feature with a classifier of interest
def computeCVAccuracyAndRMSE_StDev(featureWithClassifier):
	# Pull the classifier from the feature
	cOI = ee.Classifier(featureWithClassifier.get('c'))

	# Create a function to map through the fold assignments and compute the overall accuracy
	# for all validation folds
	def computeAccuracyForFold(foldFeature):
		# Organize the training and validation data
		foldNumber = ee.Number(ee.Feature(foldFeature).get('Fold'))
		trainingData = fcOI.filterMetadata(cvFoldString,'not_equals',foldNumber)
		validationData = fcOI.filterMetadata(cvFoldString,'equals',foldNumber)

		# Train the classifier and classify the validation dataset
#		trainedClassifier = cOI.train(trainingData,classProperty,covariateList, 0.2, 1).train(trainingData,classProperty,covariateList, 0.2, 2).train(trainingData,classProperty,covariateList, 0.2, 3).train(trainingData,classProperty,covariateList, 0.2, 4).train(trainingData,classProperty,covariateList, 0.2, 5)
		trainedClassifier = cOI.train(trainingData,classProperty,covariateList)
		outputtedPropName = classProperty+'_Predicted'
		classifiedValidationData = validationData.classify(trainedClassifier,outputtedPropName)

		# Compute accuracy metrics
		r2ToSet = coefficientOfDetermination(classifiedValidationData,classProperty,outputtedPropName)
		rmseToSet = RMSE(classifiedValidationData,classProperty,outputtedPropName)
#		maeToSet = MAE(classifiedValidationData,classProperty,outputtedPropName)
		rpiqToSet = RPIQ(classifiedValidationData,classProperty,outputtedPropName)
		return foldFeature.set('R2',r2ToSet).set('RMSE', rmseToSet).set('RPIQ', rpiqToSet)#.set('MAE', maeToSet)

	# Compute the mean and std dev of the accuracy values of the classifier across all folds
	accuracyFC = kFoldAssignmentFC.map(computeAccuracyForFold)
#	meanAccuracy = accuracyFC.aggregate_mean('R2')
	sdAccuracy = accuracyFC.aggregate_total_sd('R2')

	# Calculate mean and std dev of RMSE
#	RMSEvals = accuracyFC.aggregate_array('RMSE')
#	RMSEvalsSquared = RMSEvals.map(lambda f: ee.Number(f).multiply(f))
#	sumOfRMSEvalsSquared = RMSEvalsSquared.reduce(ee.Reducer.sum())
#	meanRMSE = ee.Number.sqrt(ee.Number(sumOfRMSEvalsSquared).divide(k))

	sdRMSE = accuracyFC.aggregate_total_sd('RMSE')

#	# Calculate mean and std dev of MAE
#	meanMAE = accuracyFC.aggregate_mean('MAE')
#	sdMAE= accuracyFC.aggregate_total_sd('MAE')

	# Calculate mean and std dev of RPIQ
#	meanRPIQ = accuracyFC.aggregate_mean('RPIQ')
	sdRPIQ= accuracyFC.aggregate_total_sd('RPIQ')

	# Compute the feature to return
#	featureToReturn = featureWithClassifier.select(['cName']).set('Mean_R2',meanAccuracy,'StDev_R2',sdAccuracy, 'Mean_RMSE',meanRMSE,'StDev_RMSE',sdRMSE, 'Mean_RPIQ',meanRPIQ,'StDev_RPIQ',sdRPIQ) # , 'Mean_MAE',meanMAE,'StDev_MAE',sdMAE
	featureToReturn = featureWithClassifier.select(['cName']).set('StDev_R2',sdAccuracy, 'StDev_RMSE',sdRMSE, 'StDev_RPIQ',sdRPIQ) # , 'Mean_MAE',meanMAE,'StDev_MAE',sdMAE
	return featureToReturn

## 4) Running the DSM pipeline

### Initialization and Upload data to GEE

In [ ]:
# Check if FC is already present, if not perform data aggregation and upload
files_present = subprocess.run(bashCommandList_ls+[assetRootGEE+'/'+projectFolder], stdout=subprocess.PIPE).stdout.decode('utf-8').split('/n')
files_present = [fileName.replace('projects/earthengine-legacy/users/', '') for fileName in files_present]



if (assetIDForCVAssignedColl in files_present):
	# Load the collection with the assigned K-Fold assignments
	fcOI = ee.FeatureCollection(assetIDForCVAssignedColl)

	print('Size of training data:', fcOI.size().getInfo())
	print('Data prep already performed. Moving on with grid search')

else:
	# Turn the folder string into an assetID and perform the folder creation
	assetIDToCreate_Folder = assetRootGEE+'/'+projectFolder

	print(assetIDToCreate_Folder,'being created...')

	# Create the folder within Earth Engine
	subprocess.run(bashCommandList_CreateFolder+[assetIDToCreate_Folder])
	while any(x in subprocess.run(bashCommandList_Detect+[assetIDToCreate_Folder],stdout=subprocess.PIPE).stdout.decode('utf-8') for x in stringsOfInterest):
		print('Waiting for asset to be created...')
		time.sleep(normalWaitTime)
	print('Asset created!')

	# Sleep to allow the server time to receive incoming requests
	time.sleep(normalWaitTime/2)

	# Specify path of raw point collection
	pathOfPointCollection = holdingFolder+'/'+titleOfRawPointCollection+'.csv'

	# Load rawPointCollection
rawPointCollection = pd.read_csv(pathOfPointCollection, sep=",")[[latString, longString, classProperty]]
print(rawPointCollection.head)
	# Print basic information on the csv
print('Number of observations in original Collection', rawPointCollection.shape[0])

	# Drop NAs
preppedCollection = rawPointCollection.dropna(how='any')
print('Number of observations after dropping NAs', preppedCollection.shape[0])

	# Rounding of "classProperty" values
if classProperty_roundTo == 0:
		preppedCollection = preppedCollection.round({classProperty: 0}).astype({classProperty:'int'})
else:
		preppedCollection = preppedCollection.round({classProperty: classProperty_roundTo})

	# Write the CSV to disk and upload it to Earth Engine as a Feature Collection
localPathToCVAssignedData = holdingFolder+'/'+titleOfCSVWithCVAssignments+'.csv'
preppedCollection.to_csv(localPathToCVAssignedData,index=False)
gcs_file_name = titleOfCSVWithCVAssignments+'.csv'

assetIDForSampling = assetRootGEE+'/'+projectFolder+'/'+titleOfCSVWithCVAssignments

#### Upload training data to Earth Engine

In [ ]:
# Upload the prepared observations to Earth Engine as a table asset
# (replaces the manual upload; the asset keeps the name expected by the Spatial filter step).
# The table is sent directly from Python. If the request exceeds the Earth Engine payload limit,
# it is uploaded in parts, the parts are merged into a single asset, and the parts are deleted.

# Number of rows per partial upload (used only if the single upload is too large)
uploadChunkSize = 5000

if asset_exists(assetIDForSampling):
	print(assetIDForSampling, 'already exists. Skipping upload.')
else:
	try:
		wait_for_task(export_fc_to_asset(df_to_fc(preppedCollection), titleOfCSVWithCVAssignments+'_upload', assetIDForSampling))
	except ee.EEException as e:
		if 'payload' not in str(e).lower():
			raise
		print('Request too large for a single upload. Uploading in parts of', uploadChunkSize, 'rows...')
		partIDs = []
		for k, startRow in enumerate(range(0, preppedCollection.shape[0], uploadChunkSize), start=1):
			partID = assetIDForSampling+'_part_'+str(k)
			if not asset_exists(partID):
				wait_for_task(export_fc_to_asset(df_to_fc(preppedCollection.iloc[startRow:startRow+uploadChunkSize]),
												 titleOfCSVWithCVAssignments+'_upload_part_'+str(k), partID))
			partIDs.append(partID)

		# Merge the parts into a single asset and delete the temporary parts
		mergedFC = ee.FeatureCollection([ee.FeatureCollection(p) for p in partIDs]).flatten()
		wait_for_task(export_fc_to_asset(mergedFC, titleOfCSVWithCVAssignments+'_upload_merge', assetIDForSampling))
		for p in partIDs:
			ee.data.deleteAsset(p)
		print('Temporary part assets deleted.')

print('Number of observations uploaded:', ee.FeatureCollection(assetIDForSampling).size().getInfo())


#### Spatial filter

In [ ]:
# Spatial filter by area of interest (using exportingGeometry)
FC = ee.FeatureCollection(assetIDForSampling)
rawPointCollection_filtered = FC.filter(ee.Filter.bounds(exportingGeometry))
print('Number of observations after spatial filtering is ', rawPointCollection_filtered.size().getInfo())

# Center the map and display the filtered dataset
Map = folium.Map(tiles='OpenStreetMap')
Map.setOptions()

Map.addLayer(rawPointCollection_filtered, {'color': 'blue'}, 'Full dataset: '+classProperty)
Map.setCenter(lon,lat,3)

Map.setControlVisibility()
Map

#### Export to Assets and local file

In [ ]:
# Export (the spatial filtered dataset) to Assets
fcFiltered_exportTask = ee.batch.Export.table.toAsset(
    collection=rawPointCollection_filtered,
    description=classProperty+'_training_data_SpatialFiltered',
    assetId=assetIDForSampling+'_SpatialFiltered'
)
fcFiltered_exportTask.start()

# Wait for a short period to ensure the command has been received by the server
time.sleep(normalWaitTime/2)

# !! Break and wait
count = 1
while count >= 1:
    taskList = [str(i) for i in ee.batch.Task.list()]
    subsetList = [s for s in taskList if titleOfCSVWithCVAssignments in s]
    subsubList = [s for s in subsetList if any(xs in s for xs in ['RUNNING', 'READY'])]
    count = len(subsubList)
    print(datetime.datetime.fromtimestamp(time.time()).strftime('%Y-%m-%d %H:%M:%S'), 'Number of running jobs:', count)
    time.sleep(normalWaitTime)

print('Moving on...')

# Load the spatial filtered dataset as a collection
fcFiltered = ee.FeatureCollection(assetIDForSampling+'_SpatialFiltered')

# Write to local file (PC)
GEE_FC_to_pd(fcFiltered).to_csv(holdingFolder+'/'+classProperty+'_training_data_SpatialFiltered.csv')

notify


### Sample dataset, pixel aggregation, assign cross validation folds

In [ ]:
# Raw dataset as FeatureCollection
pointCollection = ee.FeatureCollection(assetIDForSampling+'_SpatialFiltered')

# Sample composite
sampledFC = full_composite.reduceRegions(
    reducer = ee.Reducer.first(),
    collection = pointCollection,
    scale = 60,
    tileScale = 16)

# Instantiate list of properties to select
fullPropList = covariateList + [stratificationVariableString] + [classProperty]

# Perform pixel aggregation by converting FeatureCollection to Image
fc_asImg = ee.ImageCollection(list(map(fcToImg, fullPropList))).toBands().rename(fullPropList)

# And sampling that image to get pixel values
fc_agg = fc_asImg.sample(
    region = exportingGeometry,
    scale = 60,
    projection = 'EPSG:4326',
    geometries = True,
)

# Log transform classProperty, if specified
if log_transform_classProperty == True:
    fc_agg = fc_agg.filter(ee.Filter.gt(classProperty, 0)).map(lambda f: f.set(classProperty, ee.Number(f.get(classProperty)).log()))

# Convert biome column to int
fc_agg = fc_agg.map(lambda f: f.set(stratificationVariableString, ee.Number(f.get(stratificationVariableString)).toInt()))

# Retrieve biome classes present in dataset
biome_list = fc_agg.aggregate_array(stratificationVariableString).distinct()

# Assign folds to each feature, stratified by biome
fcToExport = ee.FeatureCollection(biome_list.map(assignFolds)).flatten()

########### Estratificação da FeatureCollection em lotes para extração das covariaveis preditoras nos pontos amostrais ###########

# Defina o número de batches (B)
B = 3

# Função para particionar a FeatureCollection em B partes
def kfoldsRandom(table, B):
    percent = 1 / B
    batches = ee.List.sequence(0, B - 1).map(lambda n:
        table.filter(ee.Filter.And(
            ee.Filter.gte('random', ee.Number(percent).multiply(n)),
            ee.Filter.lt('random', ee.Number(percent).multiply(ee.Number(n).add(1)))
        ))
    )
    return batches

# Particione a FeatureCollection em B partes
batches = kfoldsRandom(fcToExport, B)

# Adicione uma nova coluna chamada "Batch" com o valor correspondente
fcWithBatch = fcToExport.map(lambda feature: feature.set('Batch', ee.Number(feature.get('random')).toInt()))

# Exporte cada parte para ativos no GEE
for i in range(B):
    fc_part = ee.FeatureCollection(batches.get(i))
    export_task = ee.batch.Export.table.toAsset(
        collection=fc_part,
        description=f'{classProperty}_training_data_part_{i + 1}',
        assetId=f'{assetIDForCVAssignedColl}_part_{i + 1}'
    )
    export_task.start()

In [ ]:
# !! Break and wait
count = 1
while count >= 1:
	taskList = [str(i) for i in ee.batch.Task.list()]
	subsetList = [s for s in taskList if classProperty in s]
	subsubList = [s for s in subsetList if any(xs in s for xs in ['RUNNING', 'READY'])]
	count = len(subsubList)
	print(datetime.datetime.fromtimestamp(time.time()).strftime('%Y-%m-%d %H:%M:%S'), 'Number of running jobs:', count)
	time.sleep(normalWaitTime*4)
print('Moving on...')

notify

In [ ]:
# Load the collection with the pre-assigned K-Fold assignments
fcOI = ee.FeatureCollection(assetIDForCVAssignedColl+'_SpatialFiltered_export')

# Display training dataset (observations + covariates)
print('Number of observations after Pixel Aggregation is ', fcOI.size().getInfo())
GEE_FC_to_pd(fcOI)

In [ ]:
# Write to file
GEE_FC_to_pd(fcOI).to_csv(outputFolder+'/'+classProperty+'_training_data_SpatialFiltered_export.csv', index=False)

### Hyperparameter tuning

In [ ]:
# Performs grid search and export results separately

# Create the a folder to store Grid Search Results
# Turn the Grid Search string into an assetID and perform the GSF creation
assetIDToCreate_GridSearch_mean = assetRootGEE+'/'+projectFolder+'/'+classProperty+'_grid_search_results_mean'
assetIDToCreate_GridSearch_StDev = assetRootGEE+'/'+projectFolder+'/'+classProperty+'_grid_search_results_StDev'
print(assetIDToCreate_GridSearch_mean,'being created...')
print(assetIDToCreate_GridSearch_StDev,'being created...')

# Create the folder within Earth Engine
subprocess.run(bashCommandList_CreateFolder+[assetIDToCreate_GridSearch_mean])
while any(x in subprocess.run(bashCommandList_Detect+[assetIDToCreate_GridSearch_mean],stdout=subprocess.PIPE).stdout.decode('utf-8') for x in stringsOfInterest):
	print('Waiting for MEAN asset to be created...')
	time.sleep(normalWaitTime)
subprocess.run(bashCommandList_CreateFolder+[assetIDToCreate_GridSearch_StDev])
while any(x in subprocess.run(bashCommandList_Detect+[assetIDToCreate_GridSearch_StDev],stdout=subprocess.PIPE).stdout.decode('utf-8') for x in stringsOfInterest):
	print('Waiting for StDev asset to be created...')
	time.sleep(normalWaitTime)
print('Assets created!', 'Starting the Grid Search...')

In [ ]:
# Classifiers
for vps in varsPerSplit_list:
	for lp in leafPop_list:
		for mn in maxNodes_list:
#			with ee.profilePrinting():
				model_name = classProperty + '_rf_VPS' + str(vps) + '_LP' + str(lp) + '_MN' + str(mn)

				rf = ee.Feature(ee.Geometry.Point([0,0])).set('cName',model_name,'c',ee.Classifier.smileRandomForest(
				numberOfTrees = nTrees,
				variablesPerSplit = vps,
				minLeafPopulation = lp,
				bagFraction = 0.632,
				maxNodes = mn,
				seed = 42
				).setOutputMode('REGRESSION'))

#			grid_search_results = ee.FeatureCollection(assetRootGEE+'/'+projectFolder+'/Grid_search_results/'+classProperty+'_grid_search_results' + '_rf_VPS' + str(vps) + '_LP' + str(lp) + '_MN' + str(mn))

	# Make a feature collection from the k-fold assignment list
				kFoldAssignmentFC = ee.FeatureCollection(ee.List(kList).map(lambda n: ee.Feature(ee.Geometry.Point([0,0])).set('Fold',n)))

	# Perform grid search
				hyperparameter_tuning_mean = ee.FeatureCollection(rf).map(computeCVAccuracyAndRMSE_mean)
				hyperparameter_tuning_StDev = ee.FeatureCollection(rf).map(computeCVAccuracyAndRMSE_StDev)

	# Export to assets Mean values
				gridSearchExport_mean = ee.batch.Export.table.toAsset(
					collection = hyperparameter_tuning_mean,
					description = classProperty+'_grid_search_results_mean' + '_rf_VPS' + str(vps) + '_LP' + str(lp) + '_MN' + str(mn),
					assetId = assetIDToCreate_GridSearch_mean+'/'+classProperty+'_grid_search_results_mean' + '_rf_VPS' + str(vps) + '_LP' + str(lp) + '_MN' + str(mn)
					)
				gridSearchExport_mean.start()

				time.sleep(normalWaitTime) # Wait for a short period
	# Export to assets StDev values
				gridSearchExport_StDev = ee.batch.Export.table.toAsset(
					collection = hyperparameter_tuning_StDev,
					description = classProperty+'_grid_search_results_StDev' + '_rf_VPS' + str(vps) + '_LP' + str(lp) + '_MN' + str(mn),
					assetId = assetIDToCreate_GridSearch_StDev+'/'+classProperty+'_grid_search_results_StDev' + '_rf_VPS' + str(vps) + '_LP' + str(lp) + '_MN' + str(mn)
					)
				gridSearchExport_StDev.start()

				# Wait for a short period to ensure the command has been received by the server
				time.sleep(normalWaitTime/2)
print('All Grid Seach tasks started!')

In [ ]:
# Counter of Grid Search exported
count = 1
while count >= 1:
	taskList = [str(i) for i in ee.batch.Task.list()]
	subsetList = [s for s in taskList if classProperty in s]
	subsubList = [s for s in subsetList if any(xs in s for xs in ['RUNNING', 'READY'])]
	count = len(subsubList)
	print(datetime.datetime.fromtimestamp(time.time()).strftime('%Y-%m-%d %H:%M:%S'), '--> Grid Searches finished: ',(num_models*2)-count,' of ', num_models*2,'('+ str(ee.Number((((num_models*2)-count)/(num_models*2))*100).round().getInfo())+'% ready)',end='\r')
	time.sleep(normalWaitTime*2)

print('\n','All Grid Search results exported...')

notify

##### Check Grid Search results and RE-RUN failed tasks (DO NOT RUN; optional recovery step)

---



In [ ]:
# Check if Grid Search results are already present in Assets
# and automatically RE-RUN failed tasks

for vps in varsPerSplit_list:
	for lp in leafPop_list:
		for mn in maxNodes_list:

			# Grid search result path
			grid_search_result_path_mean = assetIDToCreate_GridSearch_mean+'/'+classProperty+'_grid_search_results_mean'+ '_rf_VPS' + str(vps) + '_LP' + str(lp) + '_MN' + str(mn)
			grid_search_result_path_StDev = assetIDToCreate_GridSearch_StDev+'/'+classProperty+'_grid_search_results_StDev'+ '_rf_VPS' + str(vps) + '_LP' + str(lp) + '_MN' + str(mn)

            # Check if GS is already present
			files_present_mean = subprocess.run(bashCommandList_ls+[assetIDToCreate_GridSearch_mean], stdout=subprocess.PIPE).stdout.decode('utf-8').split('\n')
			files_present_mean = [fileName_mean.replace('projects/earthengine-legacy/assets/', '') for fileName_mean in files_present_mean]

			files_present_StDev = subprocess.run(bashCommandList_ls+[assetIDToCreate_GridSearch_StDev], stdout=subprocess.PIPE).stdout.decode('utf-8').split('\n')
			files_present_StDev = [fileName_StDev.replace('projects/earthengine-legacy/assets/', '') for fileName_StDev in files_present_StDev]

# Mean Values
			if (grid_search_result_path_mean+'\r' in files_present_mean):
				print('Completed:', grid_search_result_path_mean)

			else:
				print('FAILED:', grid_search_result_path_mean)
				model_name = classProperty + '_rf_VPS' + str(vps) + '_LP' + str(lp) + '_MN' + str(mn)

				rf = ee.Feature(ee.Geometry.Point([0,0])).set('cName',model_name,'c',ee.Classifier.smileRandomForest(
				numberOfTrees = nTrees,
				variablesPerSplit = vps,
				minLeafPopulation = lp,
				bagFraction = 0.632,
				maxNodes = mn,
				seed = 42
				).setOutputMode('REGRESSION'))

#			grid_search_results = ee.FeatureCollection(assetRootGEE+'/'+projectFolder+'/Grid_search_results/'+classProperty+'_grid_search_results' + '_rf_VPS' + str(vps) + '_LP' + str(lp) + '_MN' + str(mn))

	# Make a feature collection from the k-fold assignment list
				kFoldAssignmentFC = ee.FeatureCollection(ee.List(kList).map(lambda n: ee.Feature(ee.Geometry.Point([0,0])).set('Fold',n)))

	# Perform grid search
				hyperparameter_tuning_mean = ee.FeatureCollection(rf).map(computeCVAccuracyAndRMSE_mean)

	# Export to assets Mean values
				gridSearchExport_mean = ee.batch.Export.table.toAsset(
					collection = hyperparameter_tuning_mean,
					description = classProperty+'_grid_search_results_mean' + '_rf_VPS' + str(vps) + '_LP' + str(lp) + '_MN' + str(mn),
					assetId = assetIDToCreate_GridSearch_mean+'/'+classProperty+'_grid_search_results_mean' + '_rf_VPS' + str(vps) + '_LP' + str(lp) + '_MN' + str(mn)
					)
				gridSearchExport_mean.start()

				print('It was RE-RUN:', grid_search_result_path_mean)
				# Wait for a short period to ensure the command has been received by the server
				time.sleep(normalWaitTime/2)


# StDev values
			if (grid_search_result_path_StDev+'\r' in files_present_StDev):
				print('Completed:', grid_search_result_path_StDev)

			else:
				print('FAILED:', grid_search_result_path_StDev)
				model_name = classProperty + '_rf_VPS' + str(vps) + '_LP' + str(lp) + '_MN' + str(mn)

				rf = ee.Feature(ee.Geometry.Point([0,0])).set('cName',model_name,'c',ee.Classifier.smileRandomForest(
				numberOfTrees = nTrees,
				variablesPerSplit = vps,
				minLeafPopulation = lp,
				bagFraction = 0.632,
				maxNodes = mn,
				seed = 42
				).setOutputMode('REGRESSION'))

#			grid_search_results = ee.FeatureCollection(assetRootGEE+'/'+projectFolder+'/Grid_search_results/'+classProperty+'_grid_search_results' + '_rf_VPS' + str(vps) + '_LP' + str(lp) + '_MN' + str(mn))

	# Make a feature collection from the k-fold assignment list
				kFoldAssignmentFC = ee.FeatureCollection(ee.List(kList).map(lambda n: ee.Feature(ee.Geometry.Point([0,0])).set('Fold',n)))

	# Perform grid search
				hyperparameter_tuning_StDev = ee.FeatureCollection(rf).map(computeCVAccuracyAndRMSE_StDev)

	# Export to assets StDev values
				gridSearchExport_StDev = ee.batch.Export.table.toAsset(
					collection = hyperparameter_tuning_StDev,
					description = classProperty+'_grid_search_results_StDev' + '_rf_VPS' + str(vps) + '_LP' + str(lp) + '_MN' + str(mn),
					assetId = assetIDToCreate_GridSearch_StDev+'/'+classProperty+'_grid_search_results_StDev' + '_rf_VPS' + str(vps) + '_LP' + str(lp) + '_MN' + str(mn)
					)
				gridSearchExport_StDev.start()

				print('It was RE-RUN:', grid_search_result_path_StDev)
				# Wait for a short period to ensure the command has been received by the server
				time.sleep(normalWaitTime/2)

print('\n','All Grid Search results verified...','\n','Failed tasks were RE-RUN!')

##### <font color="red">!!! RUN (the next cell) ONLY AFTER the GRID SEARCH Results have been completely exported.

In [ ]:
# Import and merge grid search results

# Mean values
grid_search_results_mean = ee.FeatureCollection([])

for vps in varsPerSplit_list:
	for lp in leafPop_list:
		for mn in maxNodes_list:

			# Grid search results as FC
			grid_search_result_path_mean = assetIDToCreate_GridSearch_mean+'/'+classProperty+'_grid_search_results_mean'+ '_rf_VPS' + str(vps) + '_LP' + str(lp) + '_MN' + str(mn)
			grid_search_result_mean = ee.FeatureCollection(grid_search_result_path_mean)
			grid_search_results_mean =  grid_search_results_mean.merge(grid_search_result_mean)

	# Get top model name
	bestModelName = grid_search_results_mean.limit(1, sort_acc_prop, False).first().get('cName')

	# Get top 10 models
	top_10Models = grid_search_results_mean.limit(10, sort_acc_prop, False).aggregate_array('cName')


# StDev values
grid_search_results_StDev = ee.FeatureCollection([])

for vps in varsPerSplit_list:
	for lp in leafPop_list:
		for mn in maxNodes_list:

			# Grid search results as FC
			grid_search_result_path_StDev = assetIDToCreate_GridSearch_StDev+'/'+classProperty+'_grid_search_results_StDev'+ '_rf_VPS' + str(vps) + '_LP' + str(lp) + '_MN' + str(mn)
			grid_search_result_StDev = ee.FeatureCollection(grid_search_result_path_StDev)
			grid_search_results_StDev =  grid_search_results_StDev.merge(grid_search_result_StDev)

In [ ]:
# Write results to file
GEE_FC_to_pd(grid_search_results_mean).sort_values(sort_acc_prop, ascending = False).to_csv(outputFolder+'/'+classProperty+'_grid_search_results_mean.csv')
GEE_FC_to_pd(grid_search_results_StDev).sort_values('StDev_R2', ascending = False).to_csv(outputFolder+'/'+classProperty+'_grid_search_results_StDev.csv')

In [ ]:
# Display grid search results - Mean values
GEE_FC_to_pd(grid_search_results_mean).sort_values(sort_acc_prop, ascending = False)

In [ ]:
# Display grid search results - StDev values
GEE_FC_to_pd(grid_search_results_StDev).sort_values('StDev_R2', ascending = False)

### Classify image

In [ ]:
# Classifier List
classifierList = []

for vps in varsPerSplit_list:
	for lp in leafPop_list:
		for mn in maxNodes_list:

			model_name = classProperty + '_rf_VPS' + str(vps) + '_LP' + str(lp) + '_MN' + str(mn)

			rf = ee.Feature(ee.Geometry.Point([0,0])).set('cName',model_name,'c',ee.Classifier.smileRandomForest(
			numberOfTrees = nTrees,
			variablesPerSplit = vps,
			minLeafPopulation = lp,
			bagFraction = 0.632,
			maxNodes = mn,
			seed = 42
			).setOutputMode('REGRESSION'))

			classifierList.append(rf)

if ensemble == False:
	# Load the best model from the classifier list
	classifier = ee.Classifier(ee.Feature(ee.FeatureCollection(classifierList).filterMetadata('cName', 'equals', bestModelName).first()).get('c'))

	# Train the classifier with the collection
	trainedClassifer = classifier.train(fcOI, classProperty, covariateList)

	# Classify the image
	classifiedImage = compositeToClassify.classify(trainedClassifer,classProperty+'_Predicted')

if ensemble == True:
	def classifyImage(classifierName):
		# Load the best model from the classifier list
		classifier = ee.Classifier(ee.Feature(ee.FeatureCollection(classifierList).filterMetadata('cName', 'equals', classifierName).first()).get('c'))

		# Train the classifier with the collection
		trainedClassifer = classifier.train(fcOI, classProperty, covariateList)

		# Classify the image
		classifiedImage = compositeToClassify.classify(trainedClassifer,classProperty+'_Predicted')

		return classifiedImage

	# Classify the images, return mean
	classifiedImage = ee.ImageCollection(top_10Models.map(classifyImage)).mean().rename(classProperty+'_PredictedEnsembleMean')

### Variable importance metrics

In [ ]:
# Select top models accoding to the amount of models
if grid_search_results_mean.size().getInfo() > 10:
	top_nModels = 10
else:
	top_nModels = grid_search_results_mean.size().getInfo()


if ensemble == False:
	# Select classifier
	classifier = ee.Classifier(ee.Feature(ee.FeatureCollection(classifierList).filterMetadata('cName', 'equals', bestModelName).first()).get('c'))

    # Train the classifier with the collection
	trainedClassifer = classifier.train(fcOI, classProperty, covariateList)

    # Get the feature importance from the trained classifier and write to a .csv file and as a bar plot as .png file
	featureImportances = trainedClassifer.explain().get('importance').getInfo()

	featureImportances = pd.DataFrame(featureImportances.items(),
                                      columns=['Variable', 'Feature_Importance']).sort_values(by='Feature_Importance',
                                                                                                ascending=False)

    # Scale values
	featureImportances['Feature_Importance'] = featureImportances['Feature_Importance'] - featureImportances['Feature_Importance'].min()
	featureImportances['Feature_Importance'] = featureImportances['Feature_Importance'] / featureImportances['Feature_Importance'].max()

if ensemble == True:
    # Instantiate empty dataframe
	featureImportances = pd.DataFrame(columns=['Variable', 'Feature_Importance'])

	for i in list(range(0,top_nModels)): # grid_search_results.size().getInfo()
		# Select classifier
		classifierName = top_10Models.get(i)
		classifier = ee.Classifier(ee.Feature(ee.FeatureCollection(classifierList).filterMetadata('cName', 'equals', classifierName).first()).get('c'))

        # Train the classifier with the collection
		trainedClassifer = classifier.train(fcOI, classProperty, covariateList)

        # Get the feature importance from the trained classifier and write to a .csv file and as a bar plot as .png file
		featureImportancesToAdd = trainedClassifer.explain().get('importance').getInfo()
		featureImportancesToAdd = pd.DataFrame(featureImportancesToAdd.items(),
                                          columns=['Variable', 'Feature_Importance']).sort_values(by='Feature_Importance',
                                                                                                    ascending=False)

        # Scale values
		featureImportancesToAdd['Feature_Importance'] = featureImportancesToAdd['Feature_Importance'] - featureImportancesToAdd['Feature_Importance'].min()
		featureImportancesToAdd['Feature_Importance'] = featureImportancesToAdd['Feature_Importance'] / featureImportancesToAdd['Feature_Importance'].max()

		featureImportances = pd.concat([featureImportances, featureImportancesToAdd])

	# Take mean across ensemble members
	featureImportances = pd.DataFrame(featureImportances.groupby('Variable').mean().to_records())

# Write to csv
	featureImportances.to_csv(outputFolder+'/'+classProperty+'_featureImportances.csv')

# Create and save plot
	#plt = featureImportances[:numberOfCovariates].sort_values('Feature_Importance', ascending = False).plot(x='Variable', y='Feature_Importance', kind='bar', legend=False,
  #          title='Feature Importances')
	#fig = plt.get_figure()
	#fig.savefig(outputFolder+'/'+classProperty+'_FeatureImportances.png', bbox_inches='tight')

	print('Variable importance metrics complete! Moving on...')

### Bootstrapping

In [ ]:
# Path to file with training data (observations + covariates)
pathOftraining_data = outputFolder+'/'+classProperty+'_training_data_SpatialFiltered_export.csv'
training_data = pd.read_csv(pathOftraining_data)#[[latString, longString, classProperty]]
# Drop index columns written by earlier versions of this script (e.g., 'Unnamed: 0')
training_data = training_data.loc[:, ~training_data.columns.str.startswith('Unnamed')]

In [ ]:
# Input the number of points to use for each bootstrap model: equal to number of observations in training dataset
bootstrapModelSize = training_data.shape[0]

# Create an empty dataframe
stratSample = training_data.head(0)

# Bootstrap sampling, add to df
for n in seedsToUseForBootstrapping:
	# Perform the subsetting
	sampleToConcat = training_data.groupby(stratificationVariableString, group_keys=False).apply(lambda x: x.sample(n=int(round((strataDict.get(x.name)/100)*bootstrapModelSize)), replace=True, random_state=n))
	sampleToConcat['bootstrapIteration'] = n
	stratSample = pd.concat([stratSample, sampleToConcat])

# Format the title of the CSV and export it to a holding location
fullLocalPath = holdingFolder+'/'+bootstrapSamples+'.csv'
stratSample.to_csv(holdingFolder+'/'+bootstrapSamples+'.csv',index=False)
gcs_file_name2 = bootstrapSamples+'.csv'
# Upload the bootstrap table to Earth Engine through a temporary Cloud Storage object:
# (1) copy the CSV to the bucket, (2) ingest it as an Earth Engine table asset,
# (3) delete the CSV from the bucket once the ingestion has finished (or failed).
# The bootstrap table has no coordinates: it is only used to train the bootstrap models,
# so it is ingested as a table without geometry.
assetIDForCVAssignedColl = assetRootGEE+'/'+projectFolder+'/'+bootstrapSamples
gcsObjectName = (GCSfolder.strip('/')+'/' if GCSfolder.strip('/') else '')+gcs_file_name2
gcsURI = 'gs://'+GCBbucketname+'/'+gcsObjectName

if asset_exists(assetIDForCVAssignedColl):
	print(assetIDForCVAssignedColl, 'already exists. Skipping upload.')
else:
	# Cloud Storage client using the Earth Engine credentials (they include the Cloud Storage scope)
	storage_client = storage.Client(project=projectIdGEE, credentials=ee.data.get_persistent_credentials())
	# Reference the bucket without reading its metadata (users with access restricted
	# to a folder of a shared bucket may not have permission to read bucket metadata)
	bucket = storage_client.bucket(GCBbucketname)

	# Warn if soft delete is enabled (deleted files would be retained and billed)
	try:
		bucket.reload()
		softDeletePolicy = getattr(bucket, 'soft_delete_policy', None)
		if softDeletePolicy is not None and (softDeletePolicy.retention_duration_seconds or 0) > 0:
			print('WARNING: soft delete is enabled on gs://'+GCBbucketname, '- the temporary file will be retained for',
				  softDeletePolicy.retention_duration_seconds/86400, 'days after deletion.')
	except Exception:
		print('Note: the soft delete policy of gs://'+GCBbucketname, 'could not be checked (no permission to read bucket metadata).')

	# Check write and delete permissions with a small test file before uploading the bootstrap table
	testBlob = bucket.blob(gcsObjectName+'.permission_test')
	testBlob.upload_from_string('test')
	testBlob.delete()
	print('Write and delete permissions confirmed in', formattedBucketOI)

	blob2 = bucket.blob(gcsObjectName)
	try:
		# (1) Copy the CSV to the bucket
		print('Uploading', gcs_file_name2, 'to', formattedBucketOI, '...')
		blob2.upload_from_filename(fullLocalPath, timeout=3600)
		print(bootstrapSamples+' uploaded to a GCSB!')

		# (2) Ingest the CSV as an Earth Engine table asset
		earthEngineUploadTableCommands = bashPrefix_EarthEngine+arglist_preEEUploadTable+[assetIDStringPrefix+assetIDForCVAssignedColl]+[gcsURI]
		uploadResult = subprocess.run(earthEngineUploadTableCommands, capture_output=True, text=True)
		print(uploadResult.stdout, uploadResult.stderr)
		taskIdMatch = re.search(r'Started upload task with ID:\s*(\S+)', uploadResult.stdout+uploadResult.stderr)
		if uploadResult.returncode != 0 or taskIdMatch is None:
			raise RuntimeError('The Earth Engine ingestion could not be started (see the message above).')
		print('Upload to EE queued!')
		wait_for_task(taskIdMatch.group(1))

	finally:
		# (3) Delete the temporary file from the bucket
		if blob2.exists():
			blob2.delete()
			print('Temporary file deleted from', formattedBucketOI)

# Wait for a short period to ensure the command has been received by the server
time.sleep(normalWaitTime/2)

In [ ]:
# !! Break and wait
count = 1
while count >= 1:
	taskList = [str(i) for i in ee.batch.Task.list()]
	subsetList = [s for s in taskList if classProperty in s]
	subsubList = [s for s in subsetList if any(xs in s for xs in ['RUNNING', 'READY'])]
	count = len(subsubList)
	print(datetime.datetime.fromtimestamp(time.time()).strftime('%Y-%m-%d %H:%M:%S'), 'Number of running jobs:', count, end='\r')
	time.sleep(normalWaitTime)
print('\n', 'Moving on...')

# Load the best model from the classifier list
classifierToBootstrap = ee.Classifier(ee.Feature(ee.FeatureCollection(classifierList).filterMetadata('cName','equals',bestModelName).first()).get('c'))

# Create empty list to store all fcs
fcList = []
# Run a for loop to create multiple bootstrap iterations
for n in seedsToUseForBootstrapping:

	# Format the title of the CSV and export it to a holding location
	# titleOfColl = bootstrapSamples+str(n).zfill(3)
	collectionPath = assetRootGEE+'/'+projectFolder+'/'+bootstrapSamples

	# Load the collection from the path
	fcToTrain = ee.FeatureCollection(collectionPath).filter(ee.Filter.eq('bootstrapIteration', n))

	# Append fc to list
	fcList.append(fcToTrain)

# Helper fucntion to train a RF classifier and classify the composite image
def bootstrapFunc(fc):
	# Train the classifier with the collection
	trainedClassifer = classifierToBootstrap.train(fc,classProperty,covariateList)

	# Classify the image
	classifiedImage = compositeToClassify.classify(trainedClassifer,classProperty+'_Predicted')

	return classifiedImage

# Reduce bootstrap images to mean
meanImage = ee.ImageCollection.fromImages(list(map(bootstrapFunc, fcList))).reduce(
	reducer = ee.Reducer.mean()
)

# Reduce bootstrap images to lower and upper CIs
upperLowerCIImage = ee.ImageCollection.fromImages(list(map(bootstrapFunc, fcList))).reduce(
	reducer = ee.Reducer.percentile([2.5,97.5],['lower','upper'])
)

# Reduce bootstrap images to standard deviation
stdDevImage = ee.ImageCollection.fromImages(list(map(bootstrapFunc, fcList))).reduce(
	reducer = ee.Reducer.stdDev()
)

# Coefficient of Variation: stdDev divided by mean
coefOfVarImage = stdDevImage.divide(meanImage).rename('Bootstrapped_CoefOfVar')

notify

### Univariate interpolation/extrapolation analysis

In [ ]:
# Create a feature collection with only the values from the image bands
fcForMinMax = fcOI.select(covariateList)

# Make a FC with the band names
fcWithBandNames = ee.FeatureCollection(ee.List(covariateList).map(lambda bandName: ee.Feature(ee.Geometry.Point([0,0])).set('BandName',bandName)))

def calcMinMax(f):
  bandBeingComputed = f.get('BandName')
  maxValueToSet = fcForMinMax.reduceColumns(ee.Reducer.minMax(),[bandBeingComputed])
  return f.set('MinValue',maxValueToSet.get('min')).set('MaxValue',maxValueToSet.get('max'))

# Map function
fcWithMinMaxValues = ee.FeatureCollection(fcWithBandNames).map(calcMinMax)

# Make two images from these values (a min and a max image)
maxValuesWNulls = fcWithMinMaxValues.toList(1000).map(lambda f: ee.Feature(f).get('MaxValue'))
maxDict = ee.Dictionary.fromLists(covariateList,maxValuesWNulls)
minValuesWNulls = fcWithMinMaxValues.toList(1000).map(lambda f: ee.Feature(f).get('MinValue'))
minDict = ee.Dictionary.fromLists(covariateList,minValuesWNulls)
minImage = minDict.toImage()
maxImage = maxDict.toImage()

totalBandsBinary = compositeToClassify.gte(minImage.select(covariateList)).And(compositeToClassify.lte(maxImage.select(covariateList)))
univariate_int_ext_image = totalBandsBinary.reduce('sum').divide(compositeToClassify.bandNames().length()).rename('univariate_pct_int_ext')

### Multivariate (PCA) interpolation/extrapolation analysis

In [ ]:
# PCA interpolation/extrapolation helper function
def assessExtrapolation(fcOfInterest, propOfVariance):
	# Compute the mean and standard deviation of each band, then standardize the point data
	meanVector = fcOfInterest.mean()
	stdVector = fcOfInterest.std()
	standardizedData = (fcOfInterest-meanVector)/stdVector

	# Then standardize the composite from which the points were sampled
	meanList = meanVector.tolist()
	stdList = stdVector.tolist()
	bandNames = list(meanVector.index)
	meanImage = ee.Image(meanList).rename(bandNames)
	stdImage = ee.Image(stdList).rename(bandNames)
	standardizedImage = compositeToClassify.subtract(meanImage).divide(stdImage)

	# Run a PCA on the point samples
	pcaOutput = PCA()
	pcaOutput.fit(standardizedData)

	# Save the cumulative variance represented by each PC
	cumulativeVariance = np.cumsum(np.round(pcaOutput.explained_variance_ratio_, decimals=4)*100)

	# Make a list of PC names for future organizational purposes
	pcNames = ['PC'+str(x) for x in range(1,fcOfInterest.shape[1]+1)]

	# Get the PC loadings as a data frame
	loadingsDF = pd.DataFrame(pcaOutput.components_,columns=[str(x)+'_Loads' for x in bandNames],index=pcNames)

	# Get the original data transformed into PC space
	transformedData = pd.DataFrame(pcaOutput.fit_transform(standardizedData,standardizedData),columns=pcNames)

	# Make principal components images, multiplying the standardized image by each of the eigenvectors
	# Collect each one of the images in a single image collection

	# First step: make an image collection wherein each image is a PC loadings image
	listOfLoadings = ee.List(loadingsDF.values.tolist())
	eePCNames = ee.List(pcNames)
	zippedList = eePCNames.zip(listOfLoadings)
	def makeLoadingsImage(zippedValue):
		return ee.Image.constant(ee.List(zippedValue).get(1)).rename(bandNames).set('PC',ee.List(zippedValue).get(0))
	loadingsImageCollection = ee.ImageCollection(zippedList.map(makeLoadingsImage))

	# Second step: multiply each of the loadings image by the standardized image and reduce it using a "sum"
	# to finalize the matrix multiplication
	def finalizePCImages(loadingsImage):
		PCName = ee.String(ee.Image(loadingsImage).get('PC'))
		return ee.Image(loadingsImage).multiply(standardizedImage).reduce('sum').rename([PCName]).set('PC',PCName)
	principalComponentsImages = loadingsImageCollection.map(finalizePCImages)

	# Choose how many principal components are of interest in this analysis based on amount of
	# variance explained
	numberOfComponents = sum(i < propOfVariance for i in cumulativeVariance)+1
	print('Number of Principal Components being used:',numberOfComponents)

	# Compute the combinations of the principal components being used to compute the 2-D convex hulls
	tupleCombinations = list(combinations(list(pcNames[0:numberOfComponents]),2))
	print('Number of Combinations being used:',len(tupleCombinations))

	# Generate convex hulls for an example of the principal components of interest
	cHullCoordsList = list()
	for c in tupleCombinations:
		firstPC = c[0]
		secondPC = c[1]
		outputCHull = ConvexHull(transformedData[[firstPC,secondPC]])
		listOfCoordinates = transformedData.loc[outputCHull.vertices][[firstPC,secondPC]].values.tolist()
		flattenedList = [val for sublist in listOfCoordinates for val in sublist]
		cHullCoordsList.append(flattenedList)

	# Reformat the image collection to an image with band names that can be selected programmatically
	pcImage = principalComponentsImages.toBands().rename(pcNames)

	# Generate an image collection with each PC selected with it's matching PC
	listOfPCs = ee.List(tupleCombinations)
	listOfCHullCoords = ee.List(cHullCoordsList)
	zippedListPCSiltCHulls = listOfPCs.zip(listOfCHullCoords)

	def makeToClassifyImages(zippedListPCSiltCHulls):
		imageToClassify = pcImage.select(ee.List(zippedListPCSiltCHulls).get(0)).set('CHullCoords',ee.List(zippedListPCSiltCHulls).get(1))
		classifiedImage = imageToClassify.rename('u','v').classify(ee.Classifier.spectralRegion([imageToClassify.get('CHullCoords')]))
		return classifiedImage

	classifedImages = ee.ImageCollection(zippedListPCSiltCHulls.map(makeToClassifyImages))
	finalImageToExport = classifedImages.sum().divide(ee.Image.constant(len(tupleCombinations)))

	return finalImageToExport

# PCA interpolation-extrapolation image
PCA_int_ext = assessExtrapolation(training_data[covariateList], propOfVariance).rename('PCA_pct_int_ext')

### Final image export

#### Create a grid of tiles and display on an interactive map

In [ ]:
# Create a grid
tileSize = 800000 # Adjust the size of the tile

grid = ee.FeatureCollection('projects/ee-geocisworldspecs-203/assets/world_simplyfied').geometry().coveringGrid(CRStoUse, tileSize)
grid = grid.filterBounds(exportingGeometry)
numTiles = grid.size().getInfo()
print('Number of tiles to export: ', numTiles)

# Create a index for each tile
indexes = ee.List(grid.aggregate_array('system:index'))
ids = ee.List.sequence(1,indexes.size()).getInfo()
idByIndex = ee.Dictionary.fromLists(indexes,ids)

def ID_f(f):
#    return f.set('tile', idByIndex.get(f.get('system:index')))
    return f.set('system:id', ee.Algorithms.String(idByIndex.get(f.get('system:index'))))
gridWithID = ee.FeatureCollection(grid).map(ID_f)

# Get the center coordinates of the BBox
centroid = exportingGeometry.bounds().centroid(1).getInfo()["coordinates"];
lat = centroid[1]; lon = centroid[0];

# Center the map and display the BBox and Grid of Tiles
Map = folium.Map(tiles='OpenStreetMap')
Map.setOptions()

Map.addLayer(exportingGeometry, {'color': '#000060'}, 'BBox')
Map.addLayer(gridWithID, {'color': 'FF0050'}, 'Grid of tiles')
Map.setCenter(lon,lat,2)

Map.setControlVisibility()
Map

#### Stack predicted images and export by Tile as an ImageCollection

In [ ]:
# Stacking predicted images
if log_transform_classProperty == True:
	finalImageToExport = ee.Image.cat(meanImage.exp().rename(classProperty+'_PredictedBootstrapMean'),
	upperLowerCIImage.exp())
else:
	finalImageToExport = ee.Image.cat(meanImage.rename(classProperty+'_PredictedBootstrapMean'),
	upperLowerCIImage)

# Turn the Image Collection string into an assetID and perform the IC creation
assetIDToCreate_Collection = assetRootGEE+'/'+projectFolder+'/'+classProperty+'_PredictedImages'
print(assetIDToCreate_Collection,'being created...')

# Create the IC within Earth Engine
subprocess.run(bashCommandList_CreateCollection+[assetIDToCreate_Collection])
while any(x in subprocess.run(bashCommandList_Detect+[assetIDToCreate_Collection],stdout=subprocess.PIPE).stdout.decode('utf-8') for x in stringsOfInterest):
	print('Waiting for ImageCollection to be created...')
	time.sleep(normalWaitTime)
print('ImageCollection created!')
print('Starting Image export tasks by tile...')

# Set prperties of the image to export
properties = {"Producer":"Your Name (your.email@example.org)", # USER SETTING: adjust your info
                  "Spatial_resolution_exported": str(export_res)+' meters',
                  "Date": str(pd.to_datetime('today').strftime("%m/%d/%Y  %I:%Mh"))}

# Mask oceans
ocean_mask = ee.Image('MERIT/DEM/v1_0_3').unmask().clip(exportingGeometry); # 90m DEM

finalImageToExport = finalImageToExport.set(properties);
finalImageToExport = ee.Image(finalImageToExport).updateMask(ocean_mask);

# Export image by Tiles as an Image Collection
# Script for GEE Code Editor --> https://code.earthengine.google.com/d82d6022c0b99b28c48ec7a8d4b8ac31?noload=1
for i in ids: # list(range(100,181)):

    tile = ee.Geometry.Polygon(gridWithID.filter(ee.Filter.eq('system:id', str(i))).geometry().getInfo()['coordinates'], None, False)

    exportTask = ee.batch.Export.image.toAsset(
        image = finalImageToExport.toFloat(),
        description = classProperty+'_Bootstrapped_MultibandImage'+'_tile-' + str(i),
        assetId = assetIDToCreate_Collection+'/'+classProperty+'_Bootstrapped_MultibandImage'+'_tile-' + str(i),
        crs = CRStoUse,
        crsTransform = '['+str(res_deg)+',0,-180,0,'+str(-res_deg)+',90]',
        region = tile,
        maxPixels = int(1e13),
        shardSize = shardSize,
        pyramidingPolicy = {".default": pyramidingPolicy}
    );
    exportTask.start()

    # Wait for a short period to ensure the command has been received by the server
    time.sleep(normalWaitTime)

print('Image export tasks started, moving on')

In [ ]:
# Counter of tiles being exported
count = 1
while count >= 1:
	taskList = [str(i) for i in ee.batch.Task.list()]
	subsetList = [s for s in taskList if classProperty in s]
	subsubList = [s for s in subsetList if any(xs in s for xs in ['RUNNING', 'READY'])]
	count = len(subsubList)
#	print(datetime.datetime.fromtimestamp(time.time()).strftime('%Y-%m-%d %H:%M:%S'), '--> Number of TILES being exported:', count, end='\r')
	print(datetime.datetime.fromtimestamp(time.time()).strftime('%Y-%m-%d %H:%M:%S'), '--> Tiles finished: ',numTiles-count,' of ', numTiles,'('+ str(ee.Number(((numTiles-count)/numTiles)*100).round().getInfo())+'% ready)',end='\r')
	time.sleep(normalWaitTime*2)
print('\n','All tiles exported...')

notify

##### <font color="red">!!! RUN (the next cell) ONLY AFTER the final image export have been completely exported

In [ ]:
# Import final image
img = ee.ImageCollection(assetIDToCreate_Collection).mosaic();
img1 = img.select(classProperty+'_PredictedBootstrapMean').rename('x');
#img2 = img.select(classProperty+'_Predicted_stdDev').rename('x');

# Get values from image
reducers = ee.Reducer.percentile([2,98], ['min','max']);

stats = img1.reduceRegion(reducer = reducers,
                              geometry = exportingGeometry,
                              scale = 5000,
                              tileScale = 16);
img_min = ee.Number(stats.get('x_min').getInfo()).format('%.2f').getInfo();
img_max = ee.Number(stats.get('x_max').getInfo()).format('%.2f').getInfo();
print('Min and Max values for image visualization')
print(classProperty+'_PredictedBootstrapMean --> ', 'PercMin: '+str(img_min),'|  PercMax: '+str(img_max))

#stats2 = img2.reduceRegion(reducer = reducers,
#                              geometry = exportingGeometry,
#                              scale = 5000,
#                              tileScale = 16);
#img_min2 = ee.Number(stats2.get('x_min').getInfo()).format('%.2f').getInfo();
#img_max2 = ee.Number(stats2.get('x_max').getInfo()).format('%.2f').getInfo();
#print(classProperty+'_Predicted_stdDev      --> ', 'PercMin: '+str(img_min2),'|  PercMax: '+str(img_max2))

# Create a palette from hex strings.
palette1 = ["#C9CE00","#D78E00","#DE2007","#B20022"]; # Clay
#palette1 = ["#B20022","#DE2007","#D78E00","#C9CE00"]; # Silt
#palette2 = ["#000004","#2C105C","#711F81","#B63679","#EE605E","#FDAE78","#FCFDBF"] # stdDev


# Center the map and display the BBox
Map = folium.Map(tiles='OpenStreetMap')
Map.setOptions()

#Map.addLayer(img2, {'min':img_min2, 'max':img_max2, 'palette':palette2}, classProperty+"_Predicted_stdDev")
Map.addLayer(img1, {'min':img_min, 'max':img_max, 'palette':palette1}, classProperty+"_PredictedBootstrapMean")
Map.setCenter(lon,lat,2)

Map.setControlVisibility()
Map

### Spatial Leave-One-Out cross validation

In [ ]:
# Specify buffer size as list, to test across multiple buffer sizes
buffer_size_list = list([10, 100, 1000, 2000, 5000]) # Define buffer size in meters (e.g., you can use Moran's I or other test to determine SAC range)

# !! NOTE: this is a fairly computatinally intensive excercise, so there are some precautions to take to ensure servers aren't overloaded
# !! This operaion SHOULD NOT be performed on the entire dataset
# Set number of random points to test
if training_data.shape[0] > 150:
	n_points = 150
else:
	n_points = training_data.shape[0]

# Set number of repetitions
n_reps = 1
nList = list(range(0,n_reps))


# Performs SLOO-CV and export results separately

# Create the a folder to store SLOO-CV Results
assetIDToCreate_SLOOcv = assetRootGEE+'/'+projectFolder+'/'+classProperty+'_sloo_cv_results'
print(assetIDToCreate_SLOOcv,'being created...')

# Create the folder within Earth Engine
subprocess.run(bashCommandList_CreateFolder+[assetIDToCreate_SLOOcv])
while any(x in subprocess.run(bashCommandList_Detect+[assetIDToCreate_SLOOcv],stdout=subprocess.PIPE).stdout.decode('utf-8') for x in stringsOfInterest):
	print('Waiting for asset to be created...')
	time.sleep(normalWaitTime)
print('Asset created!')

# Iterate SLOO-CV across buffer sizes
for i in range(len(buffer_size_list)):
#    with ee.profilePrinting():
        # Make a feature collection
        fc_toMap = ee.FeatureCollection(ee.List(nList).map(lambda n: ee.Feature(ee.Geometry.Point([0,0])).set('buffer_size',buffer_size_list[i]).set('rep',n)))

        # Helper function 1: Spatial Leave One Out cross-validation function:
        def BLOOcv(f):
            # Test feature
            testFeature = ee.FeatureCollection(f)

            # Training set: all samples not within geometry of test feature
            trainFC = fcOI.filter(ee.Filter.geometry(testFeature).Not())

            # Classifier to test
            if ensemble == True:
                classifierName = top_10Models.get(i)
                classifier = ee.Classifier(ee.Feature(ee.FeatureCollection(classifierList).filterMetadata('cName', 'equals', classifierName).first()).get('c'))
            else:
                classifierName = bestModelName
                classifier = ee.Classifier(ee.Feature(ee.FeatureCollection(classifierList).filterMetadata('cName', 'equals', classifierName).first()).get('c'))

            # Train classifier
            trainedClassifer = classifier.train(trainFC, classProperty, covariateList)

            # Apply classifier
            classified = testFeature.classify(classifier = trainedClassifer, outputName = 'predicted')

            # Get predicted value
            predicted = classified.first().get('predicted')

            # Set predicted value to feature
            return f.set('predicted', predicted).copyProperties(f)

        # Helper function 3: R2 calculation function
        def calc_R2(f):
            # Get iteration ID
            rep = f.get('rep')

            # FeatureCollection holding the buffer radius
            buffer_size = f.get('buffer_size')

            # Sample 1000 validation points from the data
            subsetData = fcOI.randomColumn(seed = rep).sort('random').limit(n_points)

            # Add the buffer around the validation data
            fc_wBuffer = subsetData.map(lambda f: f.buffer(buffer_size))

            # Add the iteration ID to the FC
            fc_toValidate = fc_wBuffer.map(lambda f: f.set('rep', rep))

            if loo_cv_wPointRemoval == True:
                # Remove points not within sampled range
                fc_withinSampledRange = fc_toValidate.map(WithinRange).filter(ee.Filter.eq('within_range', 1))

                # Apply blocked leave one out CV function
                predicted = fc_withinSampledRange.map(BLOOcv)

            if loo_cv_wPointRemoval == False:
                # Apply blocked leave one out CV function
                predicted = fc_toValidate.map(BLOOcv)

            # Calculate R2 value
            R2_val = coefficientOfDetermination(predicted, classProperty, 'predicted')

            return f.set('R2_val', R2_val)

        # Calculate R2 across range of buffer sizes
        sloo_cv = fc_toMap.map(calc_R2)

        # Export FC to assets
        bloo_cv_fc_export = ee.batch.Export.table.toAsset(
            collection = sloo_cv,
            description = classProperty+'_sloo_cv_' + str(buffer_size_list[i]) + 'm',
            assetId = assetIDToCreate_SLOOcv+'/'+classProperty+'_sloo_cv_results_'+ str(buffer_size_list[i]) + 'm'
        )

        bloo_cv_fc_export.start()

        print('SLOO-CV task started! --> Buffer size of ', buffer_size_list[i], 'm')
        time.sleep(normalWaitTime/2)

In [ ]:
# Counter of SLOO-CV being exported
num_task = len(buffer_size_list)

count = 1
while count >= 1:
    taskList = [str(i) for i in ee.batch.Task.list()]
    subsetList = [s for s in taskList if classProperty+'_sloo_cv_' in s]
    subsubList = [s for s in subsetList if any(xs in s for xs in ['RUNNING', 'READY'])]
    count = len(subsubList)
    print(datetime.datetime.fromtimestamp(time.time()).strftime('%Y-%m-%d %H:%M:%S'), '--> SLOO-CV finished: ',num_task-count,' of ', num_task,'('+ str(ee.Number(((num_task-count)/num_task)*100).round().getInfo())+'% ready)',end='\r')
    time.sleep(normalWaitTime*2)
print('\n', 'All SLOO-CV exported...')

notify

##### <font color="red">!!! RUN (the next cell) ONLY AFTER the Spatial SLOO-CV have been completely exported

In [ ]:
# Import and merge SLOO-CV results

PathToSloo_cv_results = ee.FeatureCollection([])

for i in buffer_size_list:

# Grid search results as FC
	PathToSloo_cv_path = ee.FeatureCollection(assetIDToCreate_SLOOcv+'/'+classProperty+'_sloo_cv_results_'+str(i)+'m')
	PathToSloo_cv_result = ee.FeatureCollection(PathToSloo_cv_path)
	PathToSloo_cv_results =  PathToSloo_cv_results.merge(PathToSloo_cv_result)

# Write SLOO-CV results to csv
GEE_FC_to_pd(PathToSloo_cv_results).sort_values('R2_val', ascending = False).to_csv(outputFolder+'/'+classProperty+'_sloo_cv_results.csv')
time.sleep(normalWaitTime)

# Display SLOO-CV results
print('Spatial Leave-One-Out Cross-Validation Results:')
GEE_FC_to_pd(PathToSloo_cv_results).sort_values('buffer_size', ascending = True)

# <center> ***End***